In [6]:
import hand_tracker
import numpy as np

tracker = hand_tracker.HandTracker()

landmarks = list(tracker.track())

AttributeError: 'numpy.dtypes.BoolDType' object has no attribute 'bits'

In [ ]:
opened = list(tracker.track())

In [9]:
closed = list(tracker.track())

In [ ]:
opened = np.stack(opened)[:, 0]
closed = np.stack(closed)[:, 0]

In [ ]:
import tensorflow as tf

ds_opened = tf.data.Dataset.from_tensor_slices(opened).map(
    lambda x: (tf.reshape(x, [-1]), 0))
ds_closed = tf.data.Dataset.from_tensor_slices(closed).map(
    lambda x: (tf.reshape(x, [-1]), 1))

ds = tf.data.Dataset.sample_from_datasets([
    ds_closed.repeat().shuffle(1000),
    ds_opened.repeat().shuffle(1000),
], [.5, .5]).shuffle(1000).batch(32).as_numpy_iterator()

In [21]:
batch = next(ds)

In [56]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state


class MLP(nn.Module):

    @nn.compact
    def __call__(self, x):
        for i in range(4):
            x = jax.nn.swish(nn.Dense(256)(x))
        return nn.Dense(2)(x)


mlp = MLP()
params = mlp.init(jax.random.PRNGKey(0), batch[0])

state = train_state.TrainState.create(apply_fn=mlp.apply,
                                      params=params,
                                      tx=optax.adam(learning_rate=1e-4))

In [ ]:
import pandas as pd
@jax.jit
def update_state(state: train_state.TrainState, batch):

    def loss_fn(params):
        pred = state.apply_fn(params, batch[0])
        loss = optax.softmax_cross_entropy_with_integer_labels(pred, batch[1])
        accuracy = jnp.mean(jnp.argmax(pred, axis=-1) == batch[1])
        return jnp.mean(loss), dict(acc=accuracy, loss=jnp.mean(loss))

    grads, metrics = jax.grad(loss_fn, has_aux=True)(state.params)

    return state.apply_gradients(grads=grads), metrics


l = []

for _ in range(200):
    state, m = update_state(state, next(ds))
    l.append(m)

In [ ]:
from pythonosc import udp_client


udp = udp_client.SimpleUDPClient("localhost", 1234)

for landmarks in tracker.track():
    landmarks = landmarks[0].flatten()
    pred = state.apply_fn(state.params, landmarks[None])
    pred = jax.nn.softmax(pred.flatten())

    udp.send_message("/hand", [float(pred[0])])